In [ ]:
# # 분석 과정과 핵심 결과 요약
# ## 1. 데이터와 정제
# - **Transport Atlas:** 교통 34개 저널, 논문 122,669행(분석 표본 118,985편), 저자 43,952명(네트워크), SPECTER2 임베딩 95,166편(보유율 약 79%).
# - **주요 정제:** 이름 형식 불일치를 고쳐 저자 매칭을 0.8%에서 약 61%(네트워크 저자 92% 연결)로 올렸습니다. 비연구 항목과 제목 중복은 `in_sample` 플래그로 제외했습니다.
# - **미해결:** `comm_combined`는 의미 커뮤니티와 무관(NMI 0.018)해서 쓰지 않습니다.

# ## 2. 연구 방향의 변천
# | 단계 | 시도 | 결과 |
# |---|---|---|
# | 1 | ML 확산: 네트워크 거리와 경력별 채택 hazard | 노출량 통제 후 약화, 복제 수준이라 접음 |
# | 2 | 전문가 합류 프리미엄(인용) | 선택효과 통제 후 평균 효과 소멸(0.126, p=0.18) |
# | 3 | 2023 이후 상호작용 | 약하게 남았지만 사후 분할, 2023 한 해 의존 가능성, 참신성 효과 없음 |
# | 4 | OpenAlex 고정 창 인용 재검정 | 인용 나이 아티팩트는 약화, 위 결론은 유지 |
# | 5 | 공급-부담 정렬(국가 단위) | 소속 국가 데이터가 없어 불가 |
# | 6 | **분야 갱신 메커니즘(현재 주축)** | 아래 |

# ## 3. 현재 주축: 분야 변화는 교체인가, 기존 저자의 적응인가
# **사전 탐색**
# - 주제 유효수 +33%, 저널-주제 NMI 증가가 전체와 고정 저널 15개에서 같은 방향이라 저널 추가의 산물이 아닙니다.
# - 신생 주제 41개, 쇠퇴 34개(COVID, 충전, 마이크로모빌리티 등 해석 가능).
# - 초기 분해에서 "교체 83%"가 나왔으나, 지속 저자 질량이 24%라는 가중치 때문인 동어반복에 가까웠고, 귀무 대비 분석에서 **기각**되었습니다.
# **귀무 대비 초과분 분석(순열)**
# - 지속 저자의 주제 이동은 표본 잡음을 크게 넘습니다(전환 excess 0.035~0.068).
# - 신규 대 지속의 같은 시점 차이(gap)는 전환보다 작거나 비슷합니다(비율 0.41~0.80).
# **견고성 검증**
# | 검증 | 결과 |
# |---|---|
# | 동명이인 위험군 7종 변형 | 전환 excess 84~92% 유지 |
# | 이름 대 OpenAlex 저자 ID(캐시) | ID 기반이 이름 기반의 103% (단, 2020년 이후 3년 대 3년 한정) |
# | 저널 내부 분해 | 전환의 74~88%가 저널 내부, gap은 10~17%만 저널 내부 |
# | 확증 분할(새 기간 4쌍) | 전환 excess 모두 양수, 전환 저널내부 81~109% |
# | 단일 저널 7개 | 모두 전환 excess > 0, 모두 전환 > gap |

# **메커니즘**
# - 지속 저자는 분야 신생 주제 증가의 95%를 따라갑니다(질량 24%).
# - 코호트가 오래될수록 신생 주제로 덜 가고 쇠퇴 주제를 더 떠납니다(단조, p<0.001). 총 이동량은 비슷하고 **방향이 다릅니다**.
# - 공저자 노출 효과는 동질성과 구분되지 않아 주장하지 않습니다.

# ## 4. 현재 서사
# > 분야의 주제 변화는 저널 안에서 기존 저자의 적응으로 일어난다. 신규-지속 차이는 대부분 신규 저자가 다른 저널로 들어오는 구성 효과다. 고참일수록 신생 주제와 다른 방향으로 이동한다.

# ## 5. 한계와 정직한 평가
# - **gap 저널내부 비율:** 예측은 3/4만 통과했습니다. 1995-99→2005-09 설정은 분모가 0에 가까워 비율이 폭주했고, 절대 excess로 보고해야 합니다.
# - **지속 저자는 선택된 다작 저자**이고, 일반 저자 전체로 일반화할 수 없습니다.
# - **이름 기반 식별**이 약점입니다. ID 검증은 단기 기간에만 가능했습니다.
# - **개인 TV(0.80~0.85)는 표본 잡음**이라 보고하지 않습니다.
# - **확증 분할은 견고성 검증**입니다. 같은 코퍼스를 쓰므로 독립 검증이 아니며, 탐색 경로가 길었다는 점을 논문에 밝혀야 합니다.
# - **인과 식별과 일반화(타 분야)가 없습니다.**
# - **선행 연구 확인이 안 된 상태**입니다. 이미 알려진 결론이면 기여가 줄어듭니다.

In [1]:
# ============================================================
# 최종 분석 데이터셋 구축 (F1~F7), 외부 수집 없음
# 입력 : analysis_data/{papers_master, paper_author_long, paper_ml_features, paper_semantic_features}.csv
#        (선택) transport-atlas-data/{paper_index.json, paper_emb.bin}, analysis_data/openalex/works_by_doi.jsonl
# 출력 : analysis_data/final/F1~F7 + MANIFEST.csv
# ============================================================
import re, json, warnings, hashlib
from collections import Counter
import numpy as np, pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

# ---------------- 설정 ----------------
RECOMPUTE_TOPICS = False   # False: 기존 paper_ml_features.csv의 topic150을 그대로 사용(권장, 주제 번호 보존)
                           # True : 임베딩에서 k-means 재실행(주제 번호가 달라질 수 있음)
RUN_OA_IDS = True          # 캐시된 OpenAlex 저자 ID 대응표 생성
ROOT, DATA = Path("./analysis_data"), Path("./transport-atlas-data")
FIN = ROOT / "final"; FIN.mkdir(exist_ok=True)
YMAX, K, RISK_PER_YEAR = 2025, 150, 12
PERIODS = {"P95_99": (1995, 1999), "P00_04": (2000, 2004), "P05_09": (2005, 2009), "P10_14": (2010, 2014),
           "P12_14": (2012, 2014), "P15_19": (2015, 2019), "P20_24": (2020, 2024), "P22_24": (2022, 2024)}
MAIN0, MAIN1 = "P10_14", "P20_24"
lap = lambda m: print(f"[ok] {m}")

# ============================================================
# F1. 논문표 + topic150
# ============================================================
pm = pd.read_csv(ROOT / "papers_master.csv",
                 usecols=["pid", "title", "doi_norm", "year", "venue_short", "in_sample", "n_authors", "is_solo", "cited_by_count"])
pm = pm[pm.in_sample.astype(bool) & (pm.year <= YMAX)].copy()

if not RECOMPUTE_TOPICS:
    tp = pd.read_csv(ROOT / "paper_ml_features.csv", usecols=["pid", "topic150"]).drop_duplicates("pid")
else:
    from sklearn.cluster import MiniBatchKMeans
    sf = pd.read_csv(ROOT / "paper_semantic_features.csv", usecols=["emb_idx", "pid", "year"])
    meta = json.loads((DATA / "paper_index.json").read_text(encoding="utf-8")); n, rb = meta["n_papers"], meta["row_bytes"]
    M = np.fromfile(DATA / "paper_emb.bin", dtype=np.uint8).reshape(n, rb)
    sc = M[:, :4].copy().view(np.float32).ravel(); q = M[:, 4:].copy().view(np.int8)       # scale 4바이트 + int8
    E = q.astype(np.float32) * sc[:, None]; E /= np.linalg.norm(E, axis=1, keepdims=True).clip(1e-9); del M, q
    df = sf.dropna(subset=["pid"]).astype({"pid": int}).merge(pm[["pid"]], on="pid", how="inner") \
           .drop_duplicates("emb_idx").sort_values("emb_idx").reset_index(drop=True)
    df["topic150"] = MiniBatchKMeans(K, random_state=0, batch_size=4096, n_init=3).fit_predict(E[df.emb_idx.values])
    tp = df[["pid", "topic150"]]
    print("※ 재클러스터링 결과는 기존 topic150과 번호가 다를 수 있습니다. 기존 결과와 비교하려면 RECOMPUTE_TOPICS=False를 쓰세요.")

F1 = pm.merge(tp, on="pid", how="left"); F1["has_topic"] = F1.topic150.notna()
npp = F1[F1.year >= 2000].assign(p=lambda d: np.minimum(d.year // 5 * 5, 2020)).groupby(["venue_short", "p"]).size().unstack(fill_value=0)
FIX = set(npp.index[(npp[[2000, 2005, 2010, 2015, 2020]] >= 50).all(axis=1)])     # 모든 5년 구간 50편 이상
F1["fixed_venue"] = F1.venue_short.isin(FIX)
F1.to_csv(FIN / "F1_papers.csv", index=False, encoding="utf-8-sig")
lap(f"F1 논문 {len(F1):,} | topic150 보유 {F1.has_topic.mean():.1%} | 고정 저널 {len(FIX)}개")

# ============================================================
# F2. 저자-논문 기본표 (동명이인 위험군 제외, 분수 가중치)
# ============================================================
pal = pd.read_csv(ROOT / "paper_author_long.csv", usecols=["pid", "author_pos", "name_norm", "match_status", "in_sample"], low_memory=False)
pal = pal[pal.in_sample.astype(bool) & pal.name_norm.notna()].drop(columns="in_sample") \
        .drop_duplicates(["pid", "name_norm"]).merge(F1[["pid", "year", "venue_short", "n_authors", "topic150"]], on="pid")
amb = set(pal.loc[pal.match_status == "ambiguous", "name_norm"])
mx = pal.groupby(["name_norm", "year"]).size().groupby("name_norm").max()
heavy = set(mx[mx >= RISK_PER_YEAR].index)
risk = amb | heavy
pal["is_risk"] = pal.name_norm.isin(risk)
BASE = pal[~pal.is_risk].copy()                                       # 경력 시작 계산용(주제 없는 논문 포함)
F2 = BASE.dropna(subset=["topic150"]).copy(); F2["topic150"] = F2.topic150.astype(int)
F2["w"] = 1.0 / F2.groupby("pid").name_norm.transform("size")         # 분수 가중치
F2 = F2[["pid", "name_norm", "author_pos", "year", "venue_short", "n_authors", "topic150", "w"]]
F2.to_csv(FIN / "F2_author_paper.csv", index=False, encoding="utf-8-sig")
pd.Series(sorted(risk), name="name_norm").to_frame().assign(ambiguous=lambda d: d.name_norm.isin(amb),
    heavy=lambda d: d.name_norm.isin(heavy)).to_csv(FIN / "F2_risk_names.csv", index=False)
lap(f"F2 저자-논문 {len(F2):,} (기대 335,988) | 저자 {F2.name_norm.nunique():,} (기대 140,553) | 위험군 {len(risk):,} (기대 1,431)")

# ============================================================
# F3. 저자 x 기간 x 저널 x 주제 (전환·gap·저널 내부 분해의 직접 입력)
# ============================================================
rows = []
for pname, (y0, y1) in PERIODS.items():
    g = F2[F2.year.between(y0, y1)].groupby(["name_norm", "venue_short", "topic150"]).w.sum().reset_index()
    rows.append(g.assign(period=pname))
F3 = pd.concat(rows, ignore_index=True)[["name_norm", "period", "venue_short", "topic150", "w"]]
F3.to_csv(FIN / "F3_author_period_venue_topic.csv", index=False, encoding="utf-8-sig")
lap(f"F3 {len(F3):,}행 (기간이 겹치는 설정은 같은 논문이 여러 기간에 중복 집계됨)")

# ============================================================
# F4. 저자표 (경력 시작, 코호트, 기간별 활동·지속/신규 플래그)
# ============================================================
A = BASE.groupby("name_norm").agg(first_year=("year", "min"), last_year=("year", "max"), n_papers=("pid", "nunique"),
                                   n_venues=("venue_short", "nunique")).reset_index()
A["cohort"] = np.select([A.first_year < 2000, A.first_year < 2005, A.first_year < 2010, A.first_year < 2015],
                        ["<2000", "2000-04", "2005-09", "2010-14"], "2015+")
mass = F3.groupby(["name_norm", "period"]).w.sum().unstack(fill_value=0)
for p in PERIODS: A[f"m_{p}"] = A.name_norm.map(mass[p]).fillna(0) if p in mass else 0.0
A["continuing_main"] = (A[f"m_{MAIN0}"] > 0) & (A[f"m_{MAIN1}"] > 0)
A["entrant_main"] = (A[f"m_{MAIN1}"] > 0) & (A[f"m_{MAIN0}"] == 0)
A.to_csv(FIN / "F4_authors.csv", index=False, encoding="utf-8-sig")
lap(f"F4 저자 {len(A):,} | 지속(주) {A.continuing_main.sum():,} (기대 5,065) | 신규(주) {A.entrant_main.sum():,} (기대 45,458)")

# ============================================================
# F6. 주제표: 신생·쇠퇴 지정(2005-09 vs 2015-19, 2020 이후 자료 미사용) + 대표 단어
# ============================================================
def share(y0, y1):
    s = F2[F2.year.between(y0, y1)].drop_duplicates("pid")
    return np.bincount(s.topic150, minlength=K) / len(s)
s0, s1 = share(2005, 2009), share(2015, 2019); r = (s1 + 1e-4) / (s0 + 1e-4)
EM = np.where((r >= 3) & (s1 >= .005))[0]; DEC = np.where((r <= 1 / 3) & (s0 >= .005))[0]
STOP = set("with from using based study analysis model models approach method methods case data their this that into under over towards toward between during traffic transport transportation".split())
T = F1[F1.has_topic & (F1.year >= 2005)].copy(); T["tk"] = T.title.map(lambda s: set(w for w in re.findall(r"[a-z]{4,}", str(s).lower()) if w not in STOP))
ca, nall = Counter(w for s in T.tk for w in s), len(T)
def words(k):
    g = T[T.topic150 == k]; c = Counter(w for s in g.tk for w in s)
    sc = {w: (v / len(g)) / (ca[w] / nall) for w, v in c.items() if v >= 10}
    return ", ".join(sorted(sc, key=sc.get, reverse=True)[:6])
s20 = share(2020, 2025)
F6 = pd.DataFrame({"topic150": range(K), "share_0509": s0, "share_1519": s1, "share_2025": s20,
                   "emerging": np.isin(range(K), EM), "declining": np.isin(range(K), DEC)})
F6["top_words"] = [words(k) for k in range(K)]
F6.to_csv(FIN / "F6_topics.csv", index=False, encoding="utf-8-sig")
lap(f"F6 신생 {len(EM)}개 (기대 14) | 쇠퇴 {len(DEC)}개 (기대 13)")

# ============================================================
# F5. 지속 저자표 (2010-14 -> 2020-24): 신생·쇠퇴 비중, 공저자 노출, 주 저널
# ============================================================
cont = set(A.loc[A.continuing_main, "name_norm"])
def per_author(y0, y1, tag):
    g = F2[F2.name_norm.isin(cont) & F2.year.between(y0, y1)].assign(em=lambda d: d.topic150.isin(EM), dc=lambda d: d.topic150.isin(DEC))
    g["we"], g["wd"] = g.w * g.em, g.w * g.dc
    o = g.groupby("name_norm").agg(m=("w", "sum"), we=("we", "sum"), wd=("wd", "sum"))
    return pd.DataFrame({f"m{tag}": o.m, f"em{tag}": o.we / o.m, f"dc{tag}": o.wd / o.m})
F5 = per_author(*PERIODS[MAIN0], "0").join(per_author(*PERIODS[MAIN1], "1"), how="inner").reset_index()
F5["d_em"], F5["d_dc"] = F5.em1 - F5.em0, F5.dc1 - F5.dc0
F5 = F5.merge(A[["name_norm", "first_year", "cohort"]], on="name_norm", how="left")
y0, y1 = PERIODS[MAIN0]; b0 = F2[F2.year.between(y0, y1)]
F5["team0"] = F5.name_norm.map(b0.groupby("name_norm").n_authors.mean())
mv = b0.groupby(["name_norm", "venue_short"]).w.sum().reset_index().sort_values("w", ascending=False).drop_duplicates("name_norm")
F5["main_venue0"] = F5.name_norm.map(mv.set_index("name_norm").venue_short)
# 공저자 신생노출: P0 공저자들의 (ego와 공저하지 않은) P0 논문 중 신생 주제 비율
bb = BASE[BASE.year.between(y0, y1)]
ego = bb[bb.name_norm.isin(cont)][["pid", "name_norm"]].rename(columns={"name_norm": "ego"})
pr = ego.merge(bb[["pid", "name_norm"]].rename(columns={"name_norm": "alter"}), on="pid"); pr = pr[pr.ego != pr.alter]
ap = bb.dropna(subset=["topic150"])[["name_norm", "pid", "topic150"]].rename(columns={"name_norm": "alter", "pid": "pid2"})
ap["em"] = ap.topic150.astype(int).isin(EM).astype(int)
x = pr[["ego", "alter"]].drop_duplicates().merge(ap, on="alter")
x = x.merge(ego.rename(columns={"pid": "pid2"}).assign(own=1), on=["ego", "pid2"], how="left"); x = x[x.own.isna()]
F5["coauthor_em0"] = F5.name_norm.map(x.groupby("ego").em.mean())
F5["n_coauthors0"] = F5.name_norm.map(pr.groupby("ego").alter.nunique()).fillna(0)
F5.to_csv(FIN / "F5_continuing_authors.csv", index=False, encoding="utf-8-sig")
lap(f"F5 지속 저자 {len(F5):,} (기대 5,065) | 평균 신생 {F5.em0.mean():.3f} -> {F5.em1.mean():.3f} (기대 0.058 -> 0.133)")

# ============================================================
# F7. (선택) OpenAlex 캐시 저자 ID 대응표 (저자 수가 일치하는 논문만, 위치 정렬)
# ============================================================
if RUN_OA_IDS and (ROOT / "openalex" / "works_by_doi.jsonl").exists():
    O = pd.DataFrame([{"doi_norm": (r.get("doi_norm") or "").lower(), "ids": r.get("authors") or [], "n_oa": r.get("n_oa_auth")}
                      for r in map(json.loads, (ROOT / "openalex" / "works_by_doi.jsonl").open(encoding="utf-8"))]).drop_duplicates("doi_norm")
    pmd = F1.assign(doi_norm=F1.doi_norm.str.lower()).dropna(subset=["doi_norm"]).drop_duplicates("doi_norm")[["pid", "doi_norm"]]
    O = O.merge(pmd, on="doi_norm")
    raw = pd.read_csv(ROOT / "paper_author_long.csv", usecols=["pid", "author_pos", "name_norm", "in_sample"], low_memory=False)
    raw = raw[raw.in_sample.astype(bool)].sort_values(["pid", "author_pos"]); raw["rank"] = raw.groupby("pid").cumcount()
    O["n_rows"] = O.pid.map(raw.groupby("pid").size())
    Ok = O[(O.ids.map(len) == O.n_oa) & (O.n_oa == O.n_rows)]
    L = pd.DataFrame({"pid": np.repeat(Ok.pid.values, Ok.ids.map(len).values), "oa_id": [i for l in Ok.ids for i in l],
                      "rank": np.concatenate([np.arange(len(l)) for l in Ok.ids])})
    F7 = raw.merge(L, on=["pid", "rank"])[["pid", "rank", "name_norm", "oa_id"]]
    F7.to_csv(FIN / "F7_oa_author_ids.csv", index=False, encoding="utf-8-sig")
    lap(f"F7 이름-ID 대응 {len(F7):,}행 | 논문 {F7.pid.nunique():,} (기대 약 35,594편, 2020년 이후)")

# ============================================================
# 매니페스트(재현성): 파일별 행 수, 해시
# ============================================================
rows = []
for f in sorted(FIN.glob("F*.csv")):
    d = pd.read_csv(f, nrows=None); rows.append(dict(file=f.name, rows=len(d), cols=d.shape[1], md5=hashlib.md5(f.read_bytes()).hexdigest()[:10]))
pd.DataFrame(rows).to_csv(FIN / "MANIFEST.csv", index=False); print("\n", pd.DataFrame(rows).to_string(index=False))
print("\n주제 번호 보존을 위해 RECOMPUTE_TOPICS=False로 만든 F1~F7을 '동결본'으로 보관하고, 후속 분석은 이 파일만 읽으세요.")

[ok] F1 논문 111,685 | topic150 보유 78.9% | 고정 저널 15개
[ok] F2 저자-논문 282,040 (기대 335,988) | 저자 101,842 (기대 140,553) | 위험군 1,431 (기대 1,431)
[ok] F3 291,488행 (기간이 겹치는 설정은 같은 논문이 여러 기간에 중복 집계됨)
[ok] F4 저자 140,553 | 지속(주) 5,065 (기대 5,065) | 신규(주) 45,458 (기대 45,458)
[ok] F6 신생 14개 (기대 14) | 쇠퇴 13개 (기대 13)
[ok] F5 지속 저자 5,065 (기대 5,065) | 평균 신생 0.058 -> 0.133 (기대 0.058 -> 0.133)
[ok] F7 이름-ID 대응 141,836행 | 논문 35,594 (기대 약 35,594편, 2020년 이후)

                             file   rows  cols        md5
                   F1_papers.csv 111685    12 3e63aca9cd
             F2_author_paper.csv 282040     8 4eabaeda88
               F2_risk_names.csv   1431     3 7b75897acc
F3_author_period_venue_topic.csv 291488     5 e07edf98e4
                  F4_authors.csv 140553    16 2e54eb18cd
       F5_continuing_authors.csv   5065    15 e6bc47343f
                   F6_topics.csv    150     7 8affd24c48
            F7_oa_author_ids.csv 141836     4 c134566c29

주제 번호 보존을 위해 RECOMPUTE_TOPICS=False로 만든 F1~F7을 '동

In [2]:
# # 검산 결과: 핵심 지표는 모두 일치합니다. F2 행 수만 기대값과 다릅니다

# ## 1. 판정
# | 항목 | 결과 | 기대값 | 판정 |
# |---|---|---|---|
# | F4 지속/신규(주 분석) | 5,065 / 45,458 | 5,065 / 45,458 | 일치 |
# | F5 지속 저자, 신생 비중 | 5,065, 0.058 → 0.133 | 동일 | 일치 |
# | F6 신생/쇠퇴 주제 | 14 / 13 | 14 / 13 | 일치 |
# | F2 위험군 이름 | 1,431 | 1,431 | 일치 |
# | F7 이름-ID 대응 논문 | 35,594 | 약 35,594 | 일치 |
# | F2 행 수 | 282,040 | 335,988 | 불일치 |
# | F2 저자 수 | 101,842 | 140,553 | 불일치 |
# 지속·신규 저자 수, 신생 비중, 신생·쇠퇴 주제 수가 정확히 맞으므로 **전환, gap, 메커니즘 분석에 쓴 입력은 재현된 것으로 봐도 됩니다.** 후속 분석은 이 동결본에서 시작해도 됩니다.

# ## 2. F2 불일치의 원인
# 원인은 제가 코드 설명에서 기대값을 잘못 붙인 것으로 보입니다.
# - 이전 분석 셀이 출력한 335,988행, 140,553명은 **주제 유무와 상관없이** 위험군 제외 후 전체 저자-논문표(`BASE`)의 크기입니다. F4의 저자 수 140,553이 이를 확인해 줍니다.
# - 제가 만든 F2는 `BASE`에서 `topic150`이 없는 행을 뺀 것입니다. 임베딩 보유율이 약 79%라 행이 줄어 282,040이 됩니다.
# - 따라서 F2의 기대값은 335,988이 아니라 282,040이 맞습니다. 저자 101,842명도 같은 이유입니다. 제 설명의 오류이며 데이터 문제는 아닙니다.
# 다만 이 때문에 **F4에서 "지속"과 "신규" 판정이 주제 있는 논문 기준**으로 나옵니다(F3에서 계산). 이전 분석과 같은 5,065 / 45,458이 나왔으니 동일한 정의입니다.

# ## 3. 사용 시 주의
# - **F4의 `first_year`, `n_papers`는 주제 없는 논문까지 포함한 값**이고, F2·F3의 질량 `w`는 주제 있는 논문만 반영합니다. 두 기준이 섞이지 않게 분석 목적에 맞춰 선택하세요.
# - F3는 기간이 겹치는 설정(`P10_14`와 `P12_14` 등)이 중복 집계되어 있습니다. 합산하지 마세요.
# - F7은 141,836행인데 이는 저자 위치 단위입니다. 주제 유무와 무관하게 구성되어 있어 F2와 행 수가 다른 것이 정상입니다.
# - 동결본을 보관하세요. 후속 분석 중 파일을 다시 만들면 주제 번호나 정의가 바뀔 수 있습니다.

# ## 4. 다음 단계
# 후속 분석 방향을 정해 주세요. 이 데이터셋에서 바로 가능한 후보는 아래와 같습니다.
# 1. **주제 단위 분해:** 어느 신생 주제에서 지속 저자의 기여가 큰지
# 2. **저널 단위 이질성:** 저널별 전환 크기와 코호트 구배
# 3. **이동 경로 행렬:** 쇠퇴 주제에서 신생 주제로 어떻게 가는지
# 4. **F7 활용:** 2020년 이후 단기 구간의 이름 대 ID 정밀 비교

# 가장 먼저 시도할 것은 1번이나 3번을 권합니다. 현재 서사("기존 저자의 적응")의 내용을 구체적으로 채워 주는 분석이라, 어떤 주제에서 누가 어디로 이동했는지 보여 줄 수 있습니다. 선택해 주시면 F 파일을 입력으로 하는 단일 셀을 만들어 드리겠습니다.